# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is expressed as a Croissant schema and is accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # DatasetMetadata object
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and fields.

In [ ]:
# Discover record sets using the Croissant metadata interface.
print("Available record sets in the dataset:\n")
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Number of fields: {len(rs.fields)}")
    for field in rs.fields:
        print(f"     - Field name: {field.name} (@id: {field.id}, type: {field.data_type})")
    print('-' * 60)
if len(record_sets) == 0:
    print("\nNo record sets found. If data is in a single table, check the metadata.distributions interface.")

### Example: Iterate over available records
Below we retrieve and preview the first two records from the main record set (using its `@id`).

In [ ]:
# If there is at least one record set, get its @id. Else exit.
if len(record_sets) > 0:
    main_recordset_id = record_sets[0].id
    print(f"Main record set ID: {main_recordset_id}\n")

    # Show the first 2 records
    for i, record in enumerate(dataset.records(record_set=main_recordset_id)):
        print(f"Record {i}: {record}")
        if i == 1:
            break
else:
    print("No record sets available.")

## 3. Data Extraction
Load data from all record sets into DataFrames for further analysis and exploration. All lookups are performed using the record set and field `@id`s.

In [ ]:
# Extract all record sets by @id to pandas DataFrames.
recordset_ids = [rs.id for rs in record_sets]
dataframes = {}

# Extract data for each record set
for recordset_id in recordset_ids:
    records = list(dataset.records(record_set=recordset_id))
    dataframes[recordset_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for record set: {recordset_id} \n  Shape: {dataframes[recordset_id].shape}\n")

# As example, show columns and preview head of first record set
if len(recordset_ids) > 0:
    main_rs = recordset_ids[0]
    print(f"Fields (@id) of record set '{main_rs}':\n{dataframes[main_rs].columns.tolist()}")
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common preprocessing: filtering, normalization, and grouping. We use field and record set `@id`s for all references.

In [ ]:
# Select the record set to analyze
record_set_id = main_rs  # from above
df = dataframes[record_set_id]
print(f"DataFrame shape: {df.shape}")

# Show numeric fields by inspecting the first row
numeric_field_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric fields available: {numeric_field_candidates}")
# If none detected, scan for integer/float-looking columns (sometimes dtype is 'object' from reading)
if not numeric_field_candidates:
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    numeric_field_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

# Choose a numeric field by @id; if not available, skip EDA
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]  # first numeric field
    print(f"Using numeric field '@id': {numeric_field_id}")

    # Use a reasonable threshold for EDA. If column has positive values, pick mean, else 10
    col_mean = df[numeric_field_id].mean()
    threshold = col_mean if col_mean > 0 else 10

    # Filter records
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):\n")
    display(filtered_df.head())

    # Normalize
    filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Pick a categorical/group field for grouping; guess from object/string columns
    group_field_candidates = df.select_dtypes(include=[object, 'category']).columns.tolist()
    group_field_id = None
    if group_field_candidates:
        group_field_id = group_field_candidates[0]
        print(f"\nAttempting to group by: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
        display(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("No numeric fields found for EDA.")

## 5. Visualization
Visualize the distribution and relationships between fields using matplotlib/seaborn and field `@id`s.

In [ ]:
# Example: Plot the distribution of the main numeric field (by @id)
if numeric_field_candidates:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# Example: Boxplot of numeric field grouped by first available group field
if numeric_field_candidates and group_field_id:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to explore, filter, and visualize a FAIR² dataset defined by a Croissant schema. All data access and manipulation referenced entities by their `@id` according to best practices for working with FAIR data packages.

With these approaches, you can extend to more in-depth analyses or integrate the dataset with additional resources, ensuring proper interoperability and reusability.
